# Breast Ultrasound Classification (EfficientNet-B0 + Grad-CAM)

MD. Shadhin Khan — ID: 0572310005101003
CSE, First Capital University of Bangladesh
Supervisor: Saifa Azmiri Mohona, Lecturer, CSE

Run: Runtime → Change runtime type → T4 GPU, then Run all.

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Dataset path

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/breast_cancer_ultrasound_project")
DATA_ROOT = Path("/content/data/BUSI")
OUTPUT_DIR = PROJECT_DIR / "outputs"
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/Breast_Ultrasound_Thesis")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    Path("/content/drive/MyDrive/Cancer/archive.zip"),
    DRIVE_BACKUP_DIR / "archive.zip",
    Path("/content/drive/MyDrive/archive.zip"),
]
ARCHIVE_ZIP = next((p for p in candidates if p.exists()), None)
assert ARCHIVE_ZIP is not None, "archive.zip not found - put it in MyDrive/Cancer/"
print("Dataset zip:", ARCHIVE_ZIP, f"({ARCHIVE_ZIP.stat().st_size/1e6:.1f} MB)")


## Extract dataset

In [ ]:
import zipfile

if not (DATA_ROOT / "Dataset_BUSI_with_GT").exists():
    with zipfile.ZipFile(ARCHIVE_ZIP) as z:
        z.extractall(DATA_ROOT)

DATASET_DIR = None
for p in DATA_ROOT.rglob("Dataset_BUSI_with_GT"):
    if (p / "normal").is_dir() and (p / "benign").is_dir() and (p / "malignant").is_dir():
        DATASET_DIR = p
        break
assert DATASET_DIR is not None, "Dataset_BUSI_with_GT not found after extraction"
print("DATASET_DIR =", DATASET_DIR)


## Install packages

In [ ]:
%pip install -q scikit-learn tqdm pandas matplotlib Pillow


## Project code

In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/dataset.py
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

CLASSES = ["normal", "benign", "malignant"]
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def scan_dataset(root):
    root = Path(root)
    samples = []
    stats = {}
    class_to_idx = {name: i for i, name in enumerate(CLASSES)}
    for cls in CLASSES:
        folder = root / cls
        if not folder.is_dir():
            raise FileNotFoundError(
                f"Missing class folder: {folder}. "
                "Set --data_dir to the directory containing normal, benign, malignant."
            )
        valid, masks, other = 0, 0, 0
        for p in sorted(folder.iterdir()):
            if not p.is_file():
                continue
            if "_mask" in p.stem.lower():
                masks += 1
            elif p.suffix.lower() in IMAGE_EXTS:
                samples.append((p, class_to_idx[cls]))
                valid += 1
            else:
                other += 1
        stats[cls] = {"valid_images": valid, "mask_files_excluded": masks, "other_files": other}
    return samples, stats


class BUSIDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples, self.stats = scan_dataset(root)
        self.class_to_idx = {name: i for i, name in enumerate(CLASSES)}
        if not self.samples:
            raise RuntimeError(f"No usable images found under {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, str(path)


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/model.py
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


def build_model(num_classes=3, pretrained=True):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/train.py
import argparse, csv, json, random, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score, roc_curve)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm
from dataset import BUSIDataset, CLASSES
from model import build_model

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms(img_size, train=False):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    yt, yp, probs, paths = [], [], [], []
    for x, y, p in loader:
        logits = model(x.to(device))
        pr = torch.softmax(logits, dim=1).cpu().numpy()
        probs.append(pr)
        yp.extend(pr.argmax(1).tolist())
        yt.extend(y.tolist())
        paths.extend(list(p))
    return yt, yp, np.concatenate(probs), paths


def save_evaluation(model, test_loader, device, out, classes=CLASSES):
    yt, yp, probs, paths = predict_all(model, test_loader, device)
    acc = accuracy_score(yt, yp)
    macro_f1 = f1_score(yt, yp, average="macro")
    weighted_f1 = f1_score(yt, yp, average="weighted")
    report_dict = classification_report(yt, yp, target_names=classes, output_dict=True, zero_division=0)
    (out / "classification_report.txt").write_text(
        classification_report(yt, yp, target_names=classes, zero_division=0), encoding="utf-8")

    cm = confusion_matrix(yt, yp, labels=list(range(len(classes))))
    with open(out / "confusion_matrix.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["actual\\predicted"] + classes); 
        for i, row in enumerate(cm): w.writerow([classes[i]] + row.tolist())
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes)), classes); ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title("Confusion Matrix (Test Set)")
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
    fig.colorbar(im); fig.tight_layout(); fig.savefig(out / "confusion_matrix.png", dpi=300); plt.close(fig)

    roc_auc = {}
    try:
        yt_bin = np.zeros((len(yt), len(classes))); yt_bin[np.arange(len(yt)), yt] = 1
        fig, ax = plt.subplots(figsize=(7, 6))
        for i, cls in enumerate(classes):
            if yt_bin[:, i].sum() in (0, len(yt)): continue
            fpr, tpr, _ = roc_curve(yt_bin[:, i], probs[:, i])
            auc_i = roc_auc_score(yt_bin[:, i], probs[:, i]); roc_auc[cls] = float(auc_i)
            ax.plot(fpr, tpr, label=f"{cls} (AUC = {auc_i:.3f})")
        if roc_auc:
            roc_auc["macro_average"] = float(np.mean(list(roc_auc.values())))
            ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("False Positive Rate")
            ax.set_ylabel("True Positive Rate"); ax.set_title("ROC Curves - One-vs-Rest (Test Set)")
            ax.legend(loc="lower right"); fig.tight_layout(); fig.savefig(out / "roc_curve.png", dpi=300)
        plt.close("all")
    except Exception as e:
        print("ROC skipped:", e)

    with open(out / "test_predictions.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["image_path", "true_class", "predicted_class", "correct"] +
                                      [f"prob_{c}" for c in classes])
        for path, t, p_, pr in zip(paths, yt, yp, probs):
            w.writerow([path, classes[t], classes[p_], int(t == p_)] + [f"{v:.6f}" for v in pr])

    metrics = {"test_accuracy": float(acc), "test_macro_f1": float(macro_f1),
               "test_weighted_f1": float(weighted_f1), "test_samples": len(yt),
               "per_class": {c: report_dict[c] for c in classes}, "roc_auc_ovr": roc_auc}
    return metrics


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--img_size", type=int, default=224)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--weight_decay", type=float, default=1e-4)
    p.add_argument("--label_smoothing", type=float, default=0.05)
    p.add_argument("--patience", type=int, default=8)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--output_dir", default="outputs")
    args = p.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    torch.use_deterministic_algorithms(False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)

    ds_train = BUSIDataset(args.data_dir, get_transforms(args.img_size, train=True))
    ds_eval = BUSIDataset(args.data_dir, get_transforms(args.img_size, train=False))
    print("Dataset statistics:", json.dumps(ds_train.stats, indent=2))
    labels = [y for _, y in ds_train.samples]
    indices = np.arange(len(labels))
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=args.seed, stratify=labels)
    temp_labels = [labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=args.seed, stratify=temp_labels)
    split = {"seed": args.seed, "classes": CLASSES,
             "train": sorted(map(int, train_idx)), "val": sorted(map(int, val_idx)),
             "test": sorted(map(int, test_idx)),
             "sample_paths": [str(pth) for pth, _ in ds_train.samples],
             "sample_labels": labels}
    (out / "split_indices.json").write_text(json.dumps(split, indent=2), encoding="utf-8")
    print(f"Split sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
    (out / "config.json").write_text(json.dumps(vars(args) | {"device": str(device),
        "python": sys.version, "torch": torch.__version__}, indent=2), encoding="utf-8")

    pin = device.type == "cuda"
    train_loader = DataLoader(Subset(ds_train, train_idx), batch_size=args.batch_size, shuffle=True, num_workers=args.workers, pin_memory=pin)
    val_loader = DataLoader(Subset(ds_eval, val_idx), batch_size=args.batch_size, shuffle=False, num_workers=args.workers, pin_memory=pin)
    test_loader = DataLoader(Subset(ds_eval, test_idx), batch_size=args.batch_size, shuffle=False, num_workers=args.workers, pin_memory=pin)

    counts = np.bincount([labels[i] for i in train_idx], minlength=len(CLASSES)).astype(float)
    class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32).to(device)
    print("Class weights:", class_weights.tolist())

    model = build_model(len(CLASSES), pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=args.label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    best_f1, bad_epochs = -1.0, 0
    history = {"train_loss": [], "val_macro_f1": [], "val_accuracy": []}
    for epoch in range(args.epochs):
        model.train(); running = 0.0
        for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{args.epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); optimizer.step()
            running += loss.item() * x.size(0)
        scheduler.step()
        train_loss = running / len(train_idx)
        yt, yp, _, _ = predict_all(model, val_loader, device)
        val_f1 = f1_score(yt, yp, average="macro"); val_acc = accuracy_score(yt, yp)
        history["train_loss"].append(float(train_loss)); history["val_macro_f1"].append(float(val_f1)); history["val_accuracy"].append(float(val_acc))
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_accuracy={val_acc:.4f}, val_macro_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, bad_epochs = float(val_f1), 0
            torch.save({"model_state_dict": model.state_dict(), "classes": CLASSES,
                        "img_size": args.img_size, "val_macro_f1": best_f1, "epoch": epoch + 1},
                       out / "best_model.pth")
            print("  -> saved best_model.pth")
        else:
            bad_epochs += 1
            if bad_epochs >= args.patience:
                print(f"Early stopping at epoch {epoch+1} (best val macro F1 = {best_f1:.4f})")
                break

    with open(out / "training_history.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["epoch", "train_loss", "val_accuracy", "val_macro_f1"])
        for i in range(len(history["train_loss"])):
            w.writerow([i + 1, history["train_loss"][i], history["val_accuracy"][i], history["val_macro_f1"][i]])
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(history["train_loss"], "o-", color="tab:red", label="Training loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss", color="tab:red")
    ax2 = ax1.twinx(); ax2.plot(history["val_macro_f1"], "s-", color="tab:blue", label="Validation macro F1")
    ax2.set_ylabel("Macro F1", color="tab:blue"); ax1.set_title("Training History")
    fig.tight_layout(); fig.savefig(out / "training_history.png", dpi=300); plt.close(fig)

    ckpt = torch.load(out / "best_model.pth", map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Evaluating best checkpoint (epoch {ckpt.get('epoch')}, val macro F1 {ckpt.get('val_macro_f1'):.4f}) on the test set...")
    metrics = save_evaluation(model, test_loader, device, out)
    metrics["best_val_macro_f1"] = best_f1; metrics["history"] = history
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(json.dumps({k: metrics[k] for k in ("test_accuracy", "test_macro_f1", "test_weighted_f1", "test_samples")}, indent=2))
    print("Done. Outputs in:", out.resolve())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/evaluate.py
import argparse, json
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset
from dataset import BUSIDataset, CLASSES
from model import build_model
from train import get_transforms, save_evaluation


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--checkpoint", default="outputs/best_model.pth")
    p.add_argument("--output_dir", default="outputs")
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--workers", type=int, default=2)
    a = p.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(a.checkpoint, map_location=device)
    classes = ckpt.get("classes", CLASSES); size = ckpt.get("img_size", 224)
    split = json.loads((Path(a.output_dir) / "split_indices.json").read_text())
    ds = BUSIDataset(a.data_dir, get_transforms(size, train=False))
    assert [str(p) for p, _ in ds.samples] == split["sample_paths"], "Dataset order differs from saved split!"
    loader = DataLoader(Subset(ds, split["test"]), batch_size=a.batch_size, shuffle=False,
                        num_workers=a.workers, pin_memory=(device.type == "cuda"))
    model = build_model(len(classes), pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    out = Path(a.output_dir); out.mkdir(parents=True, exist_ok=True)
    metrics = save_evaluation(model, loader, device, out, classes)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/predict.py
import argparse, torch
from PIL import Image
from torchvision import transforms
from model import build_model


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--image", required=True)
    p.add_argument("--checkpoint", default="outputs/best_model.pth")
    a = p.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(a.checkpoint, map_location=device)
    classes = ckpt.get("classes", ["normal", "benign", "malignant"])
    size = ckpt.get("img_size", 224)
    model = build_model(len(classes), pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"]); model.eval()
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    img = Image.open(a.image).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(model(tf(img).unsqueeze(0).to(device)), dim=1)[0]
    for i in probs.argsort(descending=True).tolist():
        print(f"{classes[i]}: {probs[i].item()*100:.2f}%")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/gradcam.py
import argparse, csv, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from dataset import BUSIDataset, CLASSES
from model import build_model
from train import get_transforms

TARGET_LAYER_NAME = "features[-1] (final Conv2dNormActivation, 1280 ch)"


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._fwd)
        self.h2 = target_layer.register_full_backward_hook(self._bwd)

    def _fwd(self, module, inp, out):
        self.activations = out

    def _bwd(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def close(self):
        self.h1.remove(); self.h2.remove()

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        probs = torch.softmax(logits, dim=1)[0]
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).relu()[0]
        cam = cam.detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, probs.detach().cpu().numpy()


def load_model(checkpoint, device):
    ckpt = torch.load(checkpoint, map_location=device)
    classes = ckpt.get("classes", CLASSES)
    size = ckpt.get("img_size", 224)
    model = build_model(len(classes), pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"]); model.eval()
    return model, classes, size


def save_panel(image_pil, cam, true_name, pred_name, confidence, probs, classes, output_path):
    size = image_pil.size[0]
    heat = Image.fromarray(np.uint8(cam * 255)).resize((size, size), Image.BILINEAR)
    heat_np = np.asarray(heat)
    orig_np = np.asarray(image_pil.convert("RGB").resize((size, size)))
    cmap = plt.get_cmap("jet")
    heat_rgb = (cmap(heat_np / 255.0)[:, :, :3] * 255).astype(np.uint8)
    overlay = (0.55 * orig_np + 0.45 * heat_rgb).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
    axes[0].imshow(orig_np); axes[0].set_title("Original ultrasound"); axes[0].axis("off")
    axes[1].imshow(heat_np, cmap="jet"); axes[1].set_title("Grad-CAM heatmap"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Grad-CAM overlay"); axes[2].axis("off")
    prob_txt = "  |  ".join(f"{c}: {p*100:.1f}%" for c, p in zip(classes, probs))
    fig.suptitle(f"True: {true_name}   Predicted: {pred_name}   Confidence: {confidence*100:.2f}%\n{prob_txt}",
                 fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200); plt.close(fig)


def run_single(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, classes, size = load_model(args.checkpoint, device)
    cam_gen = GradCAM(model, model.features[-1])
    image = Image.open(args.image).convert("RGB")
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    x = tf(image).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = int(torch.softmax(model(x), dim=1).argmax(1).item())
    cam, probs = cam_gen(x, pred)
    cam_gen.close()
    save_panel(image.resize((size, size)), cam, "(unknown)", classes[pred], float(probs[pred]),
               probs, classes, args.output)
    print(f"Prediction: {classes[pred]} ({probs[pred]*100:.2f}%) -> {args.output}")


def run_batch(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, classes, size = load_model(args.checkpoint, device)
    split = json.loads(Path(args.split_file).read_text())
    ds = BUSIDataset(args.data_dir, get_transforms(size, train=False))
    assert [str(p) for p, _ in ds.samples] == split["sample_paths"], "Dataset order differs from saved split!"
    test_idx = split["test"]
    cam_gen = GradCAM(model, model.features[-1])
    out_root = Path(args.output_dir)
    rows = []
    per_class_done = {c: 0 for c in classes}
    incorrect_done = 0
    for idx in test_idx:
        path, true_label = ds.samples[idx]
        image = Image.open(path).convert("RGB")
        x = get_transforms(size, train=False)(image).unsqueeze(0).to(device)
        with torch.no_grad():
            probs0 = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        pred = int(probs0.argmax()); conf = float(probs0[pred])
        correct = int(pred == true_label)
        want_class = per_class_done[classes[true_label]] < args.per_class
        want_incorrect = (not correct) and incorrect_done < args.per_class
        if not (want_class or want_incorrect or correct):
            continue
        cam, probs = cam_gen(x, pred)
        stem = Path(path).stem.replace(" ", "_")
        fname = f"{classes[true_label]}__{stem}__pred-{classes[pred]}__conf-{conf:.3f}.png"
        disp = image.resize((size, size))
        saved_to = []
        if want_class:
            pth = out_root / classes[true_label].capitalize() / fname
            save_panel(disp, cam, classes[true_label], classes[pred], conf, probs, classes, pth)
            saved_to.append(str(pth)); per_class_done[classes[true_label]] += 1
        folder = "Correct_Predictions" if correct else "Incorrect_Predictions"
        if correct or want_incorrect:
            pth = out_root / folder / fname
            save_panel(disp, cam, classes[true_label], classes[pred], conf, probs, classes, pth)
            saved_to.append(str(pth))
            if not correct: incorrect_done += 1
        rows.append({"image": str(path), "true_class": classes[true_label],
                     "predicted_class": classes[pred], "confidence": f"{conf:.6f}",
                     "correct": correct, "files": ";".join(saved_to)} |
                    {f"prob_{c}": f"{float(v):.6f}" for c, v in zip(classes, probs)})
        if all(v >= args.per_class for v in per_class_done.values()) and incorrect_done >= args.per_class:
            break
    cam_gen.close()
    out_root.mkdir(parents=True, exist_ok=True)
    with open(out_root / "gradcam_summary.csv", "w", newline="") as f:
        if rows:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print(f"Grad-CAM batch done: {len(rows)} visualisations, per-class {per_class_done}, incorrect {incorrect_done}")
    print("Summary:", out_root / "gradcam_summary.csv")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--image"); p.add_argument("--checkpoint", default="outputs/best_model.pth")
    p.add_argument("--output", default="gradcam_result.png")
    p.add_argument("--batch", action="store_true")
    p.add_argument("--data_dir"); p.add_argument("--split_file", default="outputs/split_indices.json")
    p.add_argument("--per_class", type=int, default=5)
    p.add_argument("--output_dir", default="outputs/gradcam")
    a = p.parse_args()
    if a.batch:
        if not a.data_dir: p.error("--data_dir is required with --batch")
        run_batch(a)
    elif a.image:
        run_single(a)
    else:
        p.error("Give --image for a single image or --batch for the test set")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/requirements.txt
torch>=2.2
torchvision>=0.17
numpy
pandas
scikit-learn
matplotlib
Pillow
tqdm


In [ ]:
%%writefile /content/breast_cancer_ultrasound_project/README.md
# Breast Ultrasound Classification

EfficientNet-B0 + Grad-CAM on the BUSI dataset (normal / benign / malignant).

Author: MD. Shadhin Khan, CSE, First Capital University of Bangladesh

## Files
- dataset.py - loads BUSI images, skips the _mask files
- model.py - EfficientNet-B0 with a 3-class head
- train.py - training + test evaluation
- evaluate.py - re-checks the test results from the saved model
- predict.py - classify one image
- gradcam.py - Grad-CAM heatmaps (single image or batch)

## Run
Open the notebook in Google Colab, select the T4 GPU runtime and run all cells.
The dataset zip (archive.zip) must be in the Drive folder named Cancer.

Locally:
    python train.py --data_dir <Dataset_BUSI_with_GT> --epochs 20 --batch_size 16
    python predict.py --image <image.png> --checkpoint outputs/best_model.pth


## Dataset statistics

In [ ]:
import json, sys, os
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
from pathlib import Path as _P
from PIL import Image
from dataset import scan_dataset, CLASSES

samples, stats = scan_dataset(DATASET_DIR)
corrupt = []
for path, _ in samples:
    try:
        with Image.open(path) as im:
            im.verify()
    except Exception as e:
        corrupt.append({"file": str(path), "error": str(e)})
total_valid = sum(v["valid_images"] for v in stats.values())
total_masks = sum(v["mask_files_excluded"] for v in stats.values())
summary = {"dataset_dir": str(DATASET_DIR), "classes": stats,
           "total_valid_images": total_valid, "total_mask_files_excluded": total_masks,
           "corrupted_images": corrupt}
print(json.dumps(summary, indent=2))
(OUTPUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))
print("Saved:", OUTPUT_DIR / "dataset_summary.json")


## Training

In [ ]:
import os; os.chdir(PROJECT_DIR)
!python train.py --data_dir "{DATASET_DIR}" --epochs 20 --batch_size 16 --img_size 224 --workers 2 --output_dir outputs


## Results

In [ ]:
import json
from IPython.display import Image as DispImage, display
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
print(f"Test accuracy : {metrics['test_accuracy']*100:.2f}%")
print(f"Test macro F1 : {metrics['test_macro_f1']*100:.2f}%")
print(f"Test weighted F1: {metrics['test_weighted_f1']*100:.2f}%")
print("ROC-AUC (one-vs-rest):", metrics.get("roc_auc_ovr"))
print((OUTPUT_DIR / "classification_report.txt").read_text())
display(DispImage(str(OUTPUT_DIR / "training_history.png")))
display(DispImage(str(OUTPUT_DIR / "confusion_matrix.png")))
display(DispImage(str(OUTPUT_DIR / "roc_curve.png")))


## Check results again from the saved model

In [ ]:
import os; os.chdir(PROJECT_DIR)
!python evaluate.py --data_dir "{DATASET_DIR}" --checkpoint outputs/best_model.pth --output_dir outputs


## Grad-CAM (batch)

In [ ]:
import os; os.chdir(PROJECT_DIR)
!python gradcam.py --batch --data_dir "{DATASET_DIR}" --checkpoint outputs/best_model.pth --split_file outputs/split_indices.json --per_class 5 --output_dir outputs/gradcam
import pandas as pd
df = pd.read_csv(OUTPUT_DIR / "gradcam" / "gradcam_summary.csv")
print(df[["true_class", "predicted_class", "confidence", "correct"]].to_string(index=False))


## Grad-CAM examples

In [ ]:
from IPython.display import Image as DispImage, display
import glob
for cls in ["Normal", "Benign", "Malignant"]:
    files = sorted(glob.glob(str(OUTPUT_DIR / "gradcam" / cls / "*.png")))[:2]
    print("==", cls)
    for f in files:
        display(DispImage(f))


## Single image test

In [ ]:
import json
import os; os.chdir(PROJECT_DIR)
split = json.loads((OUTPUT_DIR / "split_indices.json").read_text())
demo_image = split["sample_paths"][split["test"][0]]
print("Demo image:", demo_image)
!python predict.py --image "{demo_image}" --checkpoint outputs/best_model.pth
!python gradcam.py --image "{demo_image}" --checkpoint outputs/best_model.pth --output outputs/gradcam_single_demo.png
from IPython.display import Image as DispImage, display
display(DispImage(str(OUTPUT_DIR / "gradcam_single_demo.png")))


## Figures for the thesis

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image as PILImage
import glob

viz = PROJECT_DIR / "visualizations"; viz.mkdir(exist_ok=True)
stats = json.loads((OUTPUT_DIR / "dataset_summary.json").read_text())["classes"]
plt.figure(figsize=(7, 5))
plt.bar(list(stats.keys()), [v["valid_images"] for v in stats.values()], color=["#4C9F70", "#4C72B0", "#C44E52"])
plt.ylabel("Valid ultrasound images"); plt.title("BUSI dataset - class distribution (masks excluded)")
for i, v in enumerate(stats.values()): plt.text(i, v["valid_images"] + 8, str(v["valid_images"]), ha="center")
plt.tight_layout(); plt.savefig(viz / "dataset_distribution.png", dpi=300); plt.close()

panels = []
for cls in ["Normal", "Benign", "Malignant"]:
    fs = sorted(glob.glob(str(OUTPUT_DIR / "gradcam" / cls / "*.png")))
    if fs: panels.append(PILImage.open(fs[0]))
if panels:
    w = max(p.width for p in panels); h = sum(p.height for p in panels)
    montage = PILImage.new("RGB", (w, h), "white"); y = 0
    for p in panels: montage.paste(p, (0, y)); y += p.height
    montage.save(viz / "gradcam_comparison.png")
print("Visualisations saved in:", viz)


## Experiment report

In [ ]:
reports = PROJECT_DIR / "reports"; reports.mkdir(exist_ok=True)
m = metrics
stats = json.loads((OUTPUT_DIR / "dataset_summary.json").read_text())
lines = []
lines.append("# Experiment Report - Explainable Breast Ultrasound Classification")
lines.append("")
lines.append("**Author:** MD. Shadhin Khan (ID: 0572310005101003), CSE, First Capital University of Bangladesh")
lines.append("**Supervisor:** Saifa Azmiri Mohona, Lecturer, CSE")
lines.append("")
lines.append("## Dataset")
lines.append(f"BUSI (Al-Dhabyani et al., Data in Brief, 2020; DOI 10.1016/j.dib.2019.104863). "
             f"Valid images: {stats['total_valid_images']} (masks excluded: {stats['total_mask_files_excluded']}).")
for c, v in stats["classes"].items(): lines.append(f"- {c}: {v['valid_images']} images")
lines.append("")
lines.append("## Model")
lines.append("EfficientNet-B0, ImageNet-pretrained, fine-tuned. Input 224x224. "
             "AdamW lr 3e-4, cosine schedule, class-weighted cross-entropy with label smoothing, "
             "stratified 70/15/15 split (seed 42), best checkpoint by validation macro F1.")
lines.append("")
lines.append("## Results (test set, never used for tuning)")
lines.append(f"- Accuracy: {m['test_accuracy']*100:.2f}%")
lines.append(f"- Macro F1: {m['test_macro_f1']*100:.2f}%  |  Weighted F1: {m['test_weighted_f1']*100:.2f}%")
lines.append(f"- ROC-AUC (one-vs-rest): {m.get('roc_auc_ovr')}")
lines.append("")
lines.append("## Explainability")
lines.append("Grad-CAM on the final convolutional block (features[-1]). See 05_GradCAM_Results. "
             "Grad-CAM shows model sensitivity, not clinical reasoning.")
lines.append("")
lines.append("## Limitations")
lines.append("- Small single-source dataset, class imbalance. Image-level (not patient-level) split - "
             "patient-independent generalisation is not claimed. No external/clinical validation.")
(reports / "experiment_report.md").write_text("\n".join(lines))
print("Report written:", reports / "experiment_report.md")


## Backup to Drive

In [ ]:
import shutil

folders = ["01_Project_Code", "02_Model_Checkpoints", "03_Dataset_Statistics",
           "04_Evaluation_Results", "05_GradCAM_Results", "06_Thesis_Visualizations",
           "07_Research_Reports"]
for f in folders:
    (DRIVE_BACKUP_DIR / f).mkdir(parents=True, exist_ok=True)

def copy(src, dst):
    src = Path(src)
    if src.exists():
        shutil.copy2(src, dst)

for f in ["dataset.py", "model.py", "train.py", "evaluate.py", "predict.py", "gradcam.py", "requirements.txt", "README.md"]:
    copy(PROJECT_DIR / f, DRIVE_BACKUP_DIR / "01_Project_Code" / f)
copy(PROJECT_DIR / "Breast_Ultrasound_EfficientNetB0_GradCAM.ipynb", DRIVE_BACKUP_DIR / "01_Project_Code" / "Breast_Ultrasound_EfficientNetB0_GradCAM.ipynb")
copy(OUTPUT_DIR / "best_model.pth", DRIVE_BACKUP_DIR / "02_Model_Checkpoints" / "best_model.pth")
copy(OUTPUT_DIR / "dataset_summary.json", DRIVE_BACKUP_DIR / "03_Dataset_Statistics" / "dataset_summary.json")
for f in ["metrics.json", "classification_report.txt", "confusion_matrix.png", "confusion_matrix.csv",
          "training_history.png", "training_history.csv", "roc_curve.png", "test_predictions.csv",
          "split_indices.json", "config.json"]:
    copy(OUTPUT_DIR / f, DRIVE_BACKUP_DIR / "04_Evaluation_Results" / f)
if (OUTPUT_DIR / "gradcam").exists():
    shutil.copytree(OUTPUT_DIR / "gradcam", DRIVE_BACKUP_DIR / "05_GradCAM_Results", dirs_exist_ok=True)
for f in ["dataset_distribution.png", "gradcam_comparison.png"]:
    copy(PROJECT_DIR / "visualizations" / f, DRIVE_BACKUP_DIR / "06_Thesis_Visualizations" / f)
for f in ["confusion_matrix.png", "roc_curve.png"]:
    copy(OUTPUT_DIR / f, DRIVE_BACKUP_DIR / "06_Thesis_Visualizations" / f)
if (PROJECT_DIR / "reports").exists():
    shutil.copytree(PROJECT_DIR / "reports", DRIVE_BACKUP_DIR / "07_Research_Reports", dirs_exist_ok=True)

total = sum(1 for p in DRIVE_BACKUP_DIR.rglob("*") if p.is_file())
print("Files backed up to Drive:", total)


In [ ]:
print(f"Test accuracy    : {metrics['test_accuracy']*100:.2f}%")
print(f"Test macro F1    : {metrics['test_macro_f1']*100:.2f}%")
print(f"Test weighted F1 : {metrics['test_weighted_f1']*100:.2f}%")
print("Backup folder    :", DRIVE_BACKUP_DIR)
